## 实验4：this指针与静态成员
### this指针
实验代码：

In [1]:
#include <iostream>

In [2]:
class Counter {
public:
    void print_address() const {
        std::cout
            << "this = "
            << this
            << '\n';
    }
};

{
    Counter a;
    Counter b;

    std::cout
        << "&a = "
        << &a
        << '\n';

    a.print_address();

    std::cout
        << "&b = "
        << &b
        << '\n';

    b.print_address();
}

&a = 0x16ef346df
this = 0x16ef346df
&b = 0x16ef346de
this = 0x16ef346de


运行代码可以发现：调用 `a.print_address()` 时，`&a` 和函数内部的 `this` 地址相同；调用 `b.print_address()` 时，`&b` 和 `this` 地址相同。

因此，`this` 指向当前调用成员函数的对象。这里的 `print_address()` 是常量成员函数，所以不能通过 `this` 修改当前对象的普通成员。

---


#### 成员函数怎么知道自己要操作哪个对象？
编写：
```C++
user.set_age(20);
```
从概念上可以理解为编译器隐式传递，也就是类似：
```C++
set_age(&user, 20);
```


C风格：
```C
void user_set_age(
    User* self,
    int age
);
```

隐式存在：
```C++
this
```
所以
```C++
age_ = age;
```
概念上类似：
```C++
this -> age_ = age;
```

#### 这和后续的 C ABI 有关系
C++：
```C++
engine.process(input);
```

C ABI
```C
sdk_engine_process(
    engine,
    input
);
```

后续我们会发现：
`C++ this`和`C API opaque handle`在思想上非常接近：

```
C++

object.method()

↓

implicit this


C

function(handle)

↓

explicit object pointer
```

### 静态成员
继续给`Counter`添加：

In [3]:
class Counter1 {
public:
    Counter1() {
        ++count_;
    }

    ~Counter1() {
        --count_;
    }

    static int count() {
        return count_;
    }

private:
    static inline int count_ = 0;
};

{
    std::cout
        << Counter1::count()
        << '\n';

    Counter1 a;

    std::cout
        << Counter1::count()
        << '\n';

    {
        Counter1 b;

        std::cout
            << Counter1::count()
            << '\n';
    }

    std::cout
        << Counter1::count()
        << '\n';

}

0
1
2
1


输出：
```
0
1
2
1
```

说明：静态数据成员 `count_` **不属于某个具体对象，而是由整个类共享**。每次构造 `Counter1` 对象时计数加一，每次析构时计数减一。

静态成员函数不依赖某个对象，因此推荐通过类名调用：
```C++
Counter1::count();
```
而不需要：`a.count();`